## *Directed Results*
# **Core Analyses of Vitamin D Signatures**

This notebook presents the **directed, hypothesis-driven analyses** of transcriptomic responses to Vitamin D and its analogs.  
Unlike exploratory analyses, here we focus on **predefined questions** (core signatures, dose–response, enrichment) using the modular utilities developed in `vitd_utils`.

All constants, parameters, and paths are centralized in `vitd_utils.config`, ensuring reproducibility and consistency across analyses.

## Section 1: Imports & Config.

In [ ]:
# Allow imports from src/vitd_utils
import sys
sys.path.append("../src")

# Core project utilities
from vitd_utils import config, idsymbols, coregenes, dose, gsea, plotting, stats, dataset

# Standard scientific libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

config.FIG_DIR.mkdir(parents=True, exist_ok=True)
print("FIG_DIR:", config.FIG_DIR, "| SAVE_FIGS =", config.SAVE_FIGS)

print("Results will be saved to:", config.RESULTS_DIR)
print("Figures will be saved to:", config.FIG_DIR, "| SAVE_FIGS =", config.SAVE_FIGS)



### 1.1. Load and validate

In [ ]:
EXP  = pd.read_parquet("../data/exports/expression_matrix_clean.parquet")
META = pd.read_csv("../data/exports/signature_metadata_clean.csv")

# Estandariza columnas (sig_id, cell_id, dose, analog)
META = dataset.standardize_meta(META)
print("Columns after standardize_meta:", META.columns.tolist())
print(META.filter(regex="dose", axis=1).head(2))  # ver 'dose' presente

# Alinea por sig_id
from vitd_utils import dataset as _ds
EXP, META = _ds.align_exp_meta(EXP, META)


## 2. Gene ID ↔ Symbol Mapping

Most LINCS L1000 resources use **gene IDs** as stable identifiers, while interpretation requires **gene symbols**.  
To ensure consistency, we build a robust mapping between IDs and symbols using `vitd_utils.idsymbols`.  
This step guarantees that downstream analyses (core genes, enrichment, plotting) always have readable gene names with safe fallbacks.


In [ ]:
# Load gene metadata (example: geneinfo_beta.txt already loaded in previous steps)
gene_info = pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t")

# Build ID → symbol mapping
sym_map = idsymbols.build_symbol_map(gene_info)

# Quick check
print("Mapping size:", sym_map.shape[0])
print("Examples:\n", sym_map.head())

# Test safe fallback (ID not in map should return itself)
print("Test mapping:", idsymbols.map_symbols_or_ids(["100", "102", "999"], sym_map)[:5])

## 3. Consensus Core Genes (definition & scoring)

**Goal.** Define a robust Vitamin D “core” signature (UP/DOWN genes) that recurs across contexts (here: cell lines), and compute a single **core score** per signature capturing the balance of UP vs DOWN core genes.

**Method.**
1) Build a gene × context matrix of effects (here: mean L1000 z-scores **per cell line**).
2) For each context, take the top/bottom `N_TOP` genes and **vote-count** across contexts.
3) Select `CORE_UP_N` / `CORE_DN_N` genes using a minimum vote threshold and deterministic tie-breakers (mean |effect|).
4) Compute **core_score** for every signature:  
   `core_score = mean(z(core_UP)) − mean(z(core_DN))` (column-wise centering).

All thresholds and sizes are centralized in `vitd_utils.config`.

In [ ]:
# Sanity alignment: keep only signatures present in both matrices
common_sig = [c for c in EXP.columns if c in set(META["sig_id"])]
EXP = EXP[common_sig].copy()
META = META.loc[META["sig_id"].isin(common_sig)].copy()

# --- Build gene × cell effects (mean across signatures within each cell line)
cell_indexer = META.set_index("sig_id")["cell_id"]
effects_by_cell = EXP.T.groupby(cell_indexer).mean().T

print("effects_by_cell shape:", effects_by_cell.shape)
display(effects_by_cell.iloc[:5, :5])

## 4. Dose–Response Analysis

**Goal.** Test whether Vitamin D analogs induce a monotonic transcriptomic response as dose increases, and quantify effect sizes (slopes).

**Method.**
1. Bin doses into "low" vs "high" categories for exploratory plots (`dose.binarize_dose`).
2. Test monotonicity with **Spearman correlation** (`dose.dose_monotonicity`).
3. Estimate slopes with **OLS regression** on log10(dose) (`dose.ols_hc3`) using HC3 robust errors.
4. Summarize slopes across cell lines and visualize with **forest plots** (`plotting.forest_from_models`).


### 4.1 Prepare dose metadata

In [ ]:
assert "dose" in META.columns, "Expected 'dose' in META after standardization."

META["log_dose"] = np.log10(META["dose"])
META["dose_bin"] = META.groupby("cell_id")["dose"].transform(lambda d: dose.binarize_dose(d).values)

META[["sig_id", "cell_id", "dose", "log_dose", "dose_bin"]].head()

In [ ]:
# 1) Build gene × cell effects
effects_by_cell = dataset.effects_by_cell(EXP, META)  # genes × cell_id
print("effects_by_cell:", effects_by_cell.shape)

# 2) Consensus core sets (UP/DOWN)
cons = coregenes.build_consensus_core(
    effects_by_cell,
    top_n=config.N_TOP,
    min_votes=config.VOTE_MIN,
    target_up=config.CORE_UP_N,
    target_dn=config.CORE_DN_N,
    min_non_na=10,
)
core_up_ids = cons["core_up"]
core_dn_ids = cons["core_dn"]
print(f"[core sets] UP={len(core_up_ids)} | DOWN={len(core_dn_ids)}")

# 3) Core score for every signature (columns of EXP)
core_scores = coregenes.core_score_for_matrix(
    effects=EXP,          # genes × sig_id
    core_up=core_up_ids,  # ID list (match EXP.index)
    core_dn=core_dn_ids,
    center=True,
)

# 4) Merge to META (standardized) by sig_id
if "core_score" in META.columns:
    META = META.drop(columns=["core_score"])
META = META.merge(core_scores.rename("core_score"),
                  left_on="sig_id", right_index=True, how="left")

# Sanity check
print("Has core_score?", "core_score" in META.columns, "| nulls:", META["core_score"].isna().sum())
display(META[["sig_id","cell_id","dose","core_score"]].head())


### 4.2 Monotonicity test (Spearman ρ)

In [ ]:
# Per-cell monotonicity of core_score vs dose
mono_results = (
    META.groupby("cell_id")
        .apply(lambda sub: dose.dose_monotonicity(sub["dose"], sub["core_score"]))
        .apply(pd.Series)
        .reset_index()
)

print(mono_results)

### 4.3 Forest plot — Dose–response slopes (HC3)

We estimate the slope of the dose–response (core_score ~ log10 dose) **per cell line** using OLS with **HC3 robust errors**.  
The forest plot shows the coefficient and its 95% confidence interval (CI); a vertical dashed line at 0 represents “no trend”.  
This complements the monotonicity test by quantifying **effect size** and uncertainty.

In [ ]:
# Ensure prerequisites are available
assert "dose" in META.columns, "Dose missing — run Section 0 standardization first."
assert "core_score" in META.columns, "core_score missing — run Section 3 consensus core scoring first."

# 1) Fit OLS-HC3 models per cell line
models, labels = [], []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["dose", "core_score"]].dropna()
    # Require at least 3 samples and 2 unique dose values
    if sub["dose"].nunique(dropna=True) < 2 or len(sub) < 3:
        continue
    try:
        m = dose.ols_hc3(sub["dose"].values, sub["core_score"].values)
        models.append(m)
        labels.append(str(cell))
    except Exception:
        # Skip groups where regression fails numerically
        continue

# 2) Summarize and plot
if models:
    # Summarize slopes with robust CI and p-values
    summary_df = dose.summarize_forest(models, labels)
    display(summary_df.sort_values("coef"))

    # Optionally export summary table
    if getattr(config, "SAVE_TABLES", False):
        out_csv = config.RESULTS_DIR / "dose_response_forest_by_cell.csv"
        summary_df.to_csv(out_csv, index=False)
        print(f"[saved] {out_csv}")

    # Forest plot
    ax = plotting.forest_from_models(
        models, labels,
        title="Dose–response slope (core_score ~ log10 dose) by cell line",
        sort="coef"
    )
    plotting.savefig(filename="forest_dose_response_by_cell.png")
    plt.show()
else:
    print("[info] No groups had enough dose variation to fit OLS-HC3.")


### Interpretation — Dose–response slopes (HC3)

Across cell lines, OLS–HC3 slopes for *core_score ~ log10(dose)* are positive and statistically significant in most contexts, indicating a dose-dependent induction of the Vitamin D core response:

- **MCF7**: largest slope, narrow CI, *p* ≪ 1e-6 → strong dose dependence.
- **A549** and **PC3**: clearly positive slopes with tight CIs (***p* < 1e-5**), consistent dose dependence.
- **U2OS**: positive slope with wider CI; still significant (*p* ≈ 0.043), suggesting a weaker but present trend.
- **HA1E**: small slope, CI overlaps zero (*p* ≈ 0.15), indicating limited or context-specific dose dependence.

Overall, these results support a **monotonic, dose-responsive activation** of the Vitamin D core signature in most cell lines, with effect sizes varying by context.

---

## 5.1 Groupwise Spearman correlations (with FDR)

**Goal.** Quantify monotonic dose–response trends by computing **Spearman’s ρ** between `log10(dose)` and `core_score` **within each cell line**.  
**Multiple-testing control.** We report Benjamini–Hochberg **FDR** across cell lines to control the expected false discovery rate.

**Why Spearman?** It is rank-based and robust to non-linearity and mild outliers, complementing OLS–HC3 slope estimates.


In [ ]:
# Ensure required columns exist
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns), \
    "Missing required columns. Run Sections 3 and 4.1 first."

# Prepare a minimal, clean table
df = META[["cell_id", "log_dose", "core_score"]].dropna().copy()

# Compute Spearman by group (prefer utils; fall back to a safe inline implementation if needed)
try:
    # Expected signature: stats.spearman_by_group(df, group_col, x_col, y_col)
    spearman_df = stats.spearman_by_group(df, group_col="cell_id",
                                          x_col="log_dose", y_col="core_score")
except Exception:
    # Safe fallback: pure pandas/scipy implementation
    from scipy.stats import spearmanr

    rows = []
    for cell, sub in df.groupby("cell_id", dropna=False):
        # Require at least 3 samples and 2 unique x values
        if sub["log_dose"].nunique() < 2 or len(sub) < 3:
            rows.append({"cell_id": str(cell), "rho": np.nan, "pvalue": np.nan, "n": len(sub)})
            continue
        r, p = spearmanr(sub["log_dose"], sub["core_score"], nan_policy="omit")
        rows.append({"cell_id": str(cell), "rho": r, "pvalue": p, "n": len(sub)})
    spearman_df = pd.DataFrame(rows)

# Add BH–FDR (prefer utils; fall back to statsmodels if needed)
try:
    # Expected signature: stats.add_fdr(df, p_col, new_col="fdr", method="bh")
    spearman_df = stats.add_fdr(spearman_df, p_col="pvalue", new_col="fdr", method="bh")
except Exception:
    from statsmodels.stats.multitest import multipletests
    p = spearman_df["pvalue"].values
    mask = ~np.isnan(p)
    fdr = np.full_like(p, np.nan, dtype=float)
    if mask.sum() > 0:
        _, q, _, _ = multipletests(p[mask], alpha=0.05, method="fdr_bh")
        fdr[mask] = q
    spearman_df["fdr"] = fdr

# Nicely formatted table
spearman_df = (
    spearman_df.rename(columns={"cell_id": "label", "rho": "spearman_rho"})
               .sort_values(["fdr", "spearman_rho"], ascending=[True, False])
               .reset_index(drop=True)
)

display(spearman_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_fdr.csv"
    spearman_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")


## 5.2 Bootstrap CIs for Spearman’s ρ

**Goal.** Quantify uncertainty in the groupwise Spearman correlations using nonparametric **bootstrap** CIs.  
This complements p-values/FDR with effect-size intervals that are robust to non-normality.


In [ ]:
def _spearman_stat(xy_df: pd.DataFrame) -> float:
    # small helper that returns Spearman's rho
    from scipy.stats import spearmanr
    r, _ = spearmanr(xy_df["log_dose"], xy_df["core_score"], nan_policy="omit")
    return r

rows = []
for cell, sub in df.groupby("cell_id", dropna=False):
    sub = sub.copy()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "rho": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Prefer utils if available; otherwise fallback to a simple bootstrap
    try:
        # Expected signature: stats.bootstrap_ci(data, stat_fn, n_boot=2000, ci=0.95, seed=0)
        ci_low, ci_high, point = stats.bootstrap_ci(sub, _spearman_stat, n_boot=4000, ci=0.95, seed=0)
        rows.append({"label": str(cell), "rho": point, "ci_low": ci_low, "ci_high": ci_high, "n": n})
    except Exception:
        # Basic bootstrap fallback
        rng = np.random.default_rng(0)
        boots = []
        for _ in range(4000):
            idx = rng.integers(0, n, n)  # sample with replacement
            boots.append(_spearman_stat(sub.iloc[idx]))
        boots = np.array(boots)
        ci_low, ci_high = np.nanpercentile(boots, [2.5, 97.5])
        rows.append({"label": str(cell), "rho": _spearman_stat(sub), "ci_low": ci_low, "ci_high": ci_high, "n": n})

boot_df = pd.DataFrame(rows).sort_values("rho", ascending=False).reset_index(drop=True)
display(boot_df)

# Optional: save table
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_bootstrap_ci.csv"
    boot_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Point-interval plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_df))[::-1]
plt.hlines(y, boot_df["ci_low"], boot_df["ci_high"])
plt.plot(boot_df["rho"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_df["label"])
plt.xlabel("Spearman ρ (with 95% CI)")
plt.title("Dose–response monotonicity by cell line")
plt.tight_layout()
plt.show()

### Dose–response monotonicity (Spearman’s ρ)

Groupwise Spearman correlations between *log10(dose)* and *core_score* provided a rank-based assessment of monotonicity:

- **MCF7**, **A549**, and **PC3** showed strong positive correlations (ρ ≈ 0.55–0.63) with narrow bootstrap confidence intervals excluding zero, all highly significant after FDR correction.  
- **U2OS** displayed a weaker but still positive correlation (ρ ≈ 0.32), with wider confidence bounds; significance was retained though effect size was smaller.  
- **HA1E** showed no clear monotonicity (ρ ≈ 0.08), with a CI overlapping zero and non-significant FDR.

Overall, four of the five cell lines exhibited evidence of a dose-dependent monotonic increase in the Vitamin D core signature, with effect sizes again highest in MCF7, A549, and PC3.

## 5.3 Quick OLS slopes (polyfit)

**Goal.** Provide a simple, effect-size–oriented summary of dose–response strength in each cell line.  
We fit a straight line `core_score ~ log10(dose)` using numpy’s `polyfit`, which is fast and stable but does not provide robust errors.  
These slopes complement the HC3 regression (Section 4.3) by offering an easy-to-interpret Δy/Δx metric.


In [ ]:
# Ensure required columns are available
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns)

rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    slope = stats.fit_slope_ols(sub, x="log_dose", y="core_score")
    rows.append({"label": str(cell), "slope": slope, "n": len(sub)})

slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(slopes_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_by_cell.csv"
    slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Simple bar plot of slopes
plt.figure(figsize=(6, 3.6))
plt.barh(slopes_df["label"], slopes_df["slope"], color="steelblue")
plt.axvline(0, linestyle="--", linewidth=1, color="black")
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose)")
plt.title("Quick dose–response slopes by cell line")
plt.tight_layout()
plt.show()


### Interpretation — Quick slopes

MCF7 and A549 exhibit the steepest slopes (~0.45–0.50), indicating strong dose-dependent increases in the Vitamin D core response.  
PC3 and U2OS show moderate slopes (~0.29–0.30), consistent with weaker but still positive trends.  
HA1E displays only a minor slope (~0.13), suggesting little or no consistent dose dependence in this context.  
Overall, effect sizes align with previous analyses (Spearman and HC3 regression), reinforcing the robustness of dose–response induction across most cell lines.

## 5.4 Bootstrap CIs for OLS slopes

**Goal.** Quantify the uncertainty of quick OLS slope estimates (`core_score ~ log10 dose`) using nonparametric bootstrap confidence intervals.  
This provides robust effect-size intervals that do not rely on parametric assumptions, complementing the HC3 regression (Section 4.3).


In [ ]:
rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["log_dose", "core_score"]].dropna()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "slope": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Bootstrap CI using the utility if available
    try:
        lo, hi = stats.bootstrap_ci(stats.fit_slope_ols(sub), B=4000, alpha=0.05)
        slope_val = stats.fit_slope_ols(sub)
    except Exception:
        # Manual fallback
        rng = np.random.default_rng(0)
        boots = []
        x, y = sub["log_dose"].values, sub["core_score"].values
        for _ in range(4000):
            idx = rng.integers(0, n, n)
            slope = stats.fit_slope_ols(pd.DataFrame({"log_dose": x[idx], "core_score": y[idx]}))
            boots.append(slope)
        boots = np.array([b for b in boots if not np.isnan(b)])
        slope_val = stats.fit_slope_ols(sub)
        lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"label": str(cell), "slope": slope_val, "ci_low": lo, "ci_high": hi, "n": n})

boot_slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(boot_slopes_df)

# Optional: save
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_bootstrap_ci.csv"
    boot_slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_slopes_df))[::-1]
plt.hlines(y, boot_slopes_df["ci_low"], boot_slopes_df["ci_high"])
plt.plot(boot_slopes_df["slope"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_slopes_df["label"])
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose) with 95% CI")
plt.title("Bootstrap CIs for quick dose–response slopes")
plt.tight_layout()
plt.show()


### Interpretation — Bootstrap CIs for slopes

- MCF7 and A549 show the strongest dose–response effects, with slopes ~0.45–0.50 and CIs well above zero.  
- PC3 and U2OS also display positive slopes, though with wider intervals, indicating moderate but consistent trends.  
- HA1E’s slope is small (~0.13) with a CI overlapping zero, suggesting no reliable dose dependence in this context.  
> Together, the bootstrap intervals reinforce robust dose–dependent activation of the Vitamin D core signature in most cell lines, especially in MCF7 and A549.

---


## 6.1 Preranked gene lists for enrichment

**Goal.** Translate dose–response results into **preranked gene lists** suitable for GSEA and Enrichr.  
We rank genes by their association with the Vitamin D core response, using consensus core scoring and per-cell effects.  
This step prepares standardized inputs for enrichment pipelines, ensuring comparability across cell lines and conditions.

In [ ]:
# Use per-cell effects (from Section 3)
# effects_by_cell: genes × cell_id matrix of mean z-scores
assert "effects_by_cell" in globals(), "Run Section 3 consensus first."

# Map IDs → symbols (safe fallback to IDs if missing)
sym_map = idsymbols.build_symbol_map(pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t"))

# Create preranked lists per cell line
ranked_lists = {}
for cell in effects_by_cell.columns:
    vec = effects_by_cell[cell].dropna()
    df = gsea.make_preranked(vec, sym_map=sym_map)
    ranked_lists[cell] = df
    print(f"{cell}: {df.shape[0]} genes ranked")

# Example preview
ranked_lists["MCF7"].head(10)

# Optionally save preranked lists
if getattr(config, "SAVE_TABLES", False):
    out_dir = config.RESULTS_DIR / "preranked_lists"
    out_dir.mkdir(exist_ok=True)
    for cell, df in ranked_lists.items():
        df.to_csv(out_dir / f"preranked_{cell}.rnk", sep="\t", index=False, header=False)
    print(f"[saved] {len(ranked_lists)} preranked .rnk files in {out_dir}")

## 6.2 GSEA (resumable, per cell line)

**Goal.** Test pathway-level enrichment using preranked gene lists for each cell line.  
We use a resumable, permutation-based procedure with checkpoints (per library × group), so long runs can be paused and resumed.

In [ ]:
# Make preranked lists discoverable by the GSEA runner
# (the runner looks for gsea.gsea_preranked_by_cell / _by_analog in module globals)
gsea.gsea_preranked_by_cell = ranked_lists  # {cell: DataFrame('gene','score')}

# Define libraries (paths or in-memory dicts). Defaults come from config.
libraries = config.GSEA_LIBRARIES  # e.g., Hallmarks, Reactome GMTs under ROOT_DIR/libs

# Run resumable enrichment (uses checkpoints in config.CHECKPOINT_DIR)
enr_results = gsea.run_enrichment_axis_resumable(
    axis="cell",
    libraries=libraries,
    target_perms=config.PERM_N,
    chunk_perms=config.CHUNK_PERMS,
    min_size=15,
    max_size=500,
    random_state=config.SEED,
)

# Preview: top rows for each library × first cell
for lib_name, by_group in enr_results.items():
    # pick first non-empty group
    first_nonempty = next((g for g, df in by_group.items() if df is not None and not df.empty), None)
    if first_nonempty is None:
        print(f"[warn] No results for {lib_name} — check library path or preranked input.")
        continue
    print(f"\n[{lib_name}] example group: {first_nonempty}")
    display(by_group[first_nonempty].head())


### Interpretation — GSEA (Hallmark & Reactome)

In A549, positive enrichment scores (ES > 0) with low FDR highlight pathway activation aligned with the Vitamin D–induced direction:

- **Hallmark:** strong enrichment of **E2F targets**, **Unfolded Protein Response**, **G2M checkpoint**, **mTORC1 signaling**, and **MYC targets**, indicating proliferative/cell-cycle programs and proteostasis/translation stress consistent with a stimulated transcriptional state.
- **Reactome:** enrichment for **DNA repair/replication** modules (e.g., meiotic/homologous recombination), **chromatin remodeling** (PBAF/BAF), **metabolic rewiring** (gluconeogenesis), and **histone arginine methylation**, supporting coordinated regulation of cell-cycle and epigenetic machinery.

The **leading-edge fraction** (~0.30–0.52) suggests a substantial subset of each gene set drives the signal. Cross-cell consistency (next figure) will distinguish shared vs. context-specific programs.

---

## 6.3 Dot-plot of top enriched pathways

**Goal.** Summarize the most enriched pathways across cell lines for each library.  
The dot size encodes gene set size; color encodes −log10(FDR). Groups on the x-axis are cell lines.


In [ ]:
for lib_name, by_group in enr_results.items():
    gsea.dotplot_top(
        enr_results=by_group,
        lib_name=lib_name,
        axis="cell",
        top_n=10,
        fdr_cutoff=0.05,
        groups=["A549","HA1E","MCF7","PC3","U2OS"],
        wrap_width=80,
        vmax_cap=2.5,
        fig_width=20.0,
        left_margin=0.7,
        bottom_pad=0.33,
        point_sizes=(26,140),
        )

In [ ]:
# Count how many cell lines each term is significant in (FDR < 0.05)
consensus = []
for lib_name, by_group in enr_results.items():
    rows = []
    for cell, df in by_group.items():
        if df is None or df.empty:
            continue
        sig = df[df["fdr_bh"] < 0.05].copy()
        sig["cell_id"] = cell
        rows.append(sig[["term","ES","fdr_bh","cell_id"]])
    if not rows:
        continue
    cat = pd.concat(rows, ignore_index=True)
    tally = (cat.groupby("term")
                .agg(n_cells=("cell_id","nunique"),
                     mean_ES=("ES","mean"))
                .sort_values(["n_cells","mean_ES"], ascending=[False, False])
                .reset_index())
    tally["library"] = lib_name
    consensus.append(tally)

consensus_df = pd.concat(consensus, ignore_index=True) if consensus else pd.DataFrame()
print(consensus_df.head(20))


### Consensus pathway enrichment across cell lines

Across the Hallmark collection, **10 pathways were significantly enriched (FDR < 0.05) in at least 4 out of 5 cell lines**, indicating a high degree of cross-cell reproducibility. Among them, *UV response (early and late)*, *KRAS signaling*, *glycolysis*, *xenobiotic metabolism*, *adipogenesis*, and *mitotic spindle* reached significance in all five models, with positive enrichment scores (mean ES > 0.23) suggesting consistent activation. Stress-adaptive programs such as the *unfolded protein response* and *hypoxia* were detected in four cell lines, together with proliferative and inflammatory axes including *mTORC1 signaling*, *G2M checkpoint*, and *inflammatory response*.  

This consensus pattern points to a **core set of metabolic, proliferative, and stress-related pathways** modulated by vitamin D analogs, beyond context-specific transcriptional effects. The broad recurrence across diverse cellular backgrounds supports the existence of a conserved vitamin D transcriptional program that could represent common mechanistic drivers of its biological activity.

---

## 7. Visualization of directed results

To facilitate interpretation and ensure reproducibility, we assembled a set of publication-ready figures that summarize the main analyses. These include:  
(i) **forest plots** for dose–response slopes (OLS-HC3 estimates with confidence intervals),  
(ii) **box/strip plots** to visualize core scores across doses and cell lines, and  
(iii) **dot plots** highlighting the top enriched pathways by group.  

Together, these figures provide complementary perspectives on the transcriptional response to vitamin D analogs, enabling both quantitative comparison and biological interpretation.


In [ ]:
core_scores_df = META[["sig_id", "cell_id", "log_dose", "core_score"]].dropna()

forest_df = stats.summarize_slopes_ols_hc3(core_scores_df)
ax = plotting.forest_slopes(forest_df, title="Dose–response slopes by cell line", sort="coef")
plotting.savefig(filename="forest_slopes.png")

### Dose–response slopes (HC3 regression)

Across cell lines, OLS–HC3 regression of *core_score ~ log10(dose)* revealed consistently positive slopes, indicating a monotonic activation of the Vitamin D core signature:

- **MCF7** showed the steepest slope (~0.55) with a narrow confidence interval and extremely significant *p*-value (*p* < 1e-8).  
- **A549** and **PC3** exhibited robust positive slopes (~0.40–0.45), also highly significant (*p* < 1e-5).  
- **U2OS** displayed a moderate slope (~0.28), significant but with wider uncertainty (*p* ≈ 0.043).  
- **HA1E** showed only a small, non-significant slope (~0.13; CI overlapping zero, *p* ≈ 0.15).

Together, these results indicate that four out of five tested cell lines display a statistically reliable dose–dependent increase in the Vitamin D core response, with effect sizes varying by cellular context.

###  7.2 Box + strip plots — core scores by dose and cell line

To visualize the distribution of Vitamin D core responses by dose and context, 
we plotted core scores stratified by **low vs high dose** within each cell line.  
Boxplots summarize the central tendency and variability, while overlaid strip 
points show individual signatures. This representation highlights both 
systematic trends and intra-group variability.


In [ ]:
# 7.2 Box + strip plots — core scores by dose and cell line

ax = plotting.box_strip(
    META, x="dose_bin", y="core_score", hue="cell_id",
    title="Core scores across doses and cell lines"
)
plotting.savefig(filename="box_strip_core_scores.png")
plt.show()

#### Interpretation - Core scores across doses and cell lines

Box/strip plots showed that core scores were consistently higher at **high dose** 
compared to **low dose** across most cell lines:

- **MCF7, A549, and PC3**: marked upward shift in distributions at high dose, 
  consistent with the strong positive slopes observed in regression analyses.  
- **U2OS**: modest increase in median core score, with broader within-group variability.  
- **HA1E**: overlapping distributions between low and high dose, with only a minor shift.

These visualizations confirm that the **dose-dependent induction** of the Vitamin D core 
signature is evident at the distribution level, particularly in MCF7, A549, and PC3.


### 7.3 Dot plots — top enriched pathways

To summarize the enrichment analyses, we generated dot plots of the **top pathways** 
from Hallmark and Reactome collections. In these plots, dot size reflects the gene set size, 
and color encodes −log10(FDR). The x-axis shows cell lines, and the y-axis lists 
the most significantly enriched pathways.

This visualization highlights both **shared programs** (pathways enriched in multiple 
cell lines) and **context-specific signals**.

In [ ]:
from importlib import reload
reload(plotting)

for lib_name, by_group in enr_results.items():
    ax = plotting.dotplot_top(
        by_group,
        top_n=10,
        fdr_cutoff=0.05,
        groups=["A549","HA1E","MCF7","PC3","U2OS"],
        wrap_width=80,
        vmax_cap=2.5,
        fig_width=14,
        point_sizes=(26, 140),
        title=f"Enrichment summary — {lib_name} by cell"
    )
    plotting.savefig(filename=f"dotplot_{lib_name.lower()}_by_cell.png")
    plt.show()


### Enriched pathways across cell lines

Dot plots of top enriched pathways from **Hallmarks** and **Reactome** collections 
highlight both shared and context-specific programs.

- **Hallmarks**: Multiple cell-cycle and metabolic programs were consistently 
  enriched across lines, including *E2F targets*, *G2M checkpoint*, *mTORC1 signaling*, 
  *MYC targets*, and *glycolysis*. Stress-adaptive modules (*unfolded protein response*, 
  *hypoxia*) and signaling axes (*KRAS*, *TNFα/NFκB*, *estrogen response*) were also 
  recurrent. These results suggest a broad activation of proliferative, metabolic, 
  and stress-related pathways under vitamin D analog treatment.  

- **Reactome**: Enrichment was more heterogeneous, with strong signals in DNA repair 
  (*meiotic recombination*, *homologous recombination*, *ATR/replication stress*), 
  chromatin remodeling (*Polycomb/PBAF complexes*), and metabolic rewiring (*gluconeogenesis*). 
  In addition, signaling pathways (e.g., *Ephrin*, *ERBB2*, *Ras/FGFR*) and immune-related 
  modules (*NFE2L2 antioxidant response*, *IL-10 signaling*) emerged in specific contexts.  

Together, these pathway-level results indicate that vitamin D analogs elicit both 
**conserved transcriptional programs** (cell-cycle, metabolism, stress adaptation) 
and **context-dependent responses** (DNA repair, chromatin, immune signaling), 
reflecting the diversity of cellular backgrounds.
